In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import re
warnings.filterwarnings("ignore")

ROOT = Path("/home/jupyter/project")

MODEL_PATH = ROOT / "model" / "final_model.joblib"
FEATURES_PATH = ROOT / "model" / "feature_names.json"
SOURCE_CSV = ROOT / "eeg_subject_level_features.csv"

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD = 0.5
RANDOM_STATE = 42
df = pd.read_csv(SOURCE_CSV)

with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    FEATURE_NAMES = json.load(f)

model = joblib.load(MODEL_PATH)

print("Subjects:", len(df))
print("Features:", len(FEATURE_NAMES))
print(df["group"].value_counts())

Subjects: 216
Features: 252
group
Норма            128
Соматоформные     63
ПТСР              25
Name: count, dtype: int64


Определение возраста

In [2]:
def extract_age_from_id(subject_id):
    s = str(subject_id)
    m = re.search(r"_(\d{2})$", s)

    if m:
        return int(m.group(1))

    return np.nan


df["age"] = df["subject_id"].apply(extract_age_from_id)

age_check = (
    df.groupby("group")["age"]
    .agg(
        n="size",
        age_known=lambda x: x.notna().sum(),
        age_missing=lambda x: x.isna().sum()
    )
    .reset_index()
)

display(age_check)

,group,n,age_known,age_missing
0,Норма,128,128,0
1,ПТСР,25,0,25
2,Соматоформные,63,0,63


Возрастные подгруппы

In [3]:
def age_group(age):
    if pd.isna(age):
        return np.nan

    age = int(age)
    if age < 40:
        return "<40"

    if age < 50:
        return "40–49"

    if age < 60:
        return "50–59"

    if age < 65:
        return "60–64"

    return "65+"

df["age_group"] = df["age"].apply(age_group)

age_distribution = (
    df.dropna(subset=["age"])
      .groupby(["group", "age_group"])
      .size()
      .reset_index(name="n")
)

display(age_distribution)

,group,age_group,n
0,Норма,40–49,12
1,Норма,50–59,10
2,Норма,65+,21
3,Норма,<40,85


Предсказание ПСТР 

In [4]:
X = df[FEATURE_NAMES].copy() #берем признаки в том же порядке, в котором они использовались при обучении
#приводим признаки к числовому формату
for col in FEATURE_NAMES:
    X[col] = pd.to_numeric(X[col], errors="coerce")
#получаем вероятность ПТСР
df["ptsd_probability"] = model.predict_proba(X)[:, 1]
#при пороге 0.5 получаем бинарный прогноз
df["ptsd_prediction"] = (
    df["ptsd_probability"] >= 0.5
).astype(int)

display(
    df[
        ["subject_id", "group", "age", "age_group",
         "ptsd_probability", "ptsd_prediction"]
    ].head()
)

,subject_id,group,age,age_group,ptsd_probability,ptsd_prediction
0,A003_68,Норма,68.0,65+,1.000000,1
1,A036_69,Норма,69.0,65+,0.000037,0
2,A037_69,Норма,69.0,65+,0.999922,1
3,A038_67,Норма,67.0,65+,0.999677,1
4,А004_68,Норма,68.0,65+,0.999285,1


Метрики по возрастным подгруппам

In [5]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

age_results = []

for age_group_name, subset in df.groupby(
    "age_group",
    dropna=True
):

    y = (subset["group"] == "ПТСР").astype(int).to_numpy()
    p = subset["ptsd_probability"].to_numpy()
    pred = (p >= 0.5).astype(int)

    ptsd_n = int(y.sum())
    control_n = int((y == 0).sum())

    result = {
        "age_group": age_group_name,
        "n": len(subset),
        "ptsd_n": ptsd_n,
        "control_n": control_n,
        "roc_auc": np.nan,
        "balanced_accuracy": np.nan,
        "specificity": np.nan,
        "false_positive_rate": np.nan,
    }

    #ROC-AUC и BA только если присутствуют оба класса
    if ptsd_n > 0 and control_n > 0:

        result["roc_auc"] = roc_auc_score(y, p)

        result["balanced_accuracy"] = (
            balanced_accuracy_score(y, pred)
        )

    #Специфичность среди контролей
    if control_n > 0:

        control_pred = pred[y == 0]

        fp = int(control_pred.sum())
        tn = control_n - fp

        result["specificity"] = tn / control_n
        result["false_positive_rate"] = fp / control_n

    age_results.append(result)


age_metrics = pd.DataFrame(age_results)

display(age_metrics)

age_metrics.to_csv(
    OUTPUT_DIR / "confounder_age_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

,age_group,n,ptsd_n,control_n,roc_auc,balanced_accuracy,specificity,false_positive_rate
0,40–49,12,0,12,NaN,NaN,0.833333,0.166667
1,50–59,10,0,10,NaN,NaN,1.000000,0.000000
2,65+,21,0,21,NaN,NaN,0.047619,0.952381
3,<40,85,0,85,NaN,NaN,0.694118,0.305882


При стратификации контрольной группы по возрасту наблюдалась выраженная неоднородность специфичности модели. В подгруппе участников 60+ специфичность составила 4.8%, а доля ложноположительных классификаций ПТСР — 95.2% (20 из 21 участника). Поскольку возраст отсутствует для участников группы ПТСР, полноценная оценка диагностической эффективности модели внутри возрастных страт невозможна. Полученный результат указывает на необходимость дополнительного анализа возрастного конфаундинга.

Возрастной конфаундинг-анализ: можно ли по ЭЭГ-признакам предсказать возраст внутри контрольной групп?

Если ЭЭГ хорошо предсказывает возраст, это не доказывает наличие конфаундинга ПТСР-модели, но является важным диагностическим сигналом: ЭЭГ действительно содержит возраст-связанную информацию.

In [6]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [ ]:
#проверяем, можно ли предсказать возраст по тем же признакам, которые использует модель ПТСР
controls = df[
    (df["group"] == "Норма") &
    df["age"].notna()
].copy()
#используем те же признаки, которые подавались в модель ПТСР
X_age = controls[FEATURE_NAMES].copy()

for col in FEATURE_NAMES:
    X_age[col] = pd.to_numeric(X_age[col], errors="coerce")

y_age = controls["age"].to_numpy()
#модель для предсказания возраста
age_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ))
])

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

age_predictions = np.full(len(controls), np.nan)

for train_idx, test_idx in cv.split(X_age):

    age_model.fit(
        X_age.iloc[train_idx],
        y_age[train_idx]
    )

    age_predictions[test_idx] = age_model.predict(
        X_age.iloc[test_idx]
    )

age_mae = mean_absolute_error(
    y_age,
    age_predictions
)

age_r2 = r2_score(
    y_age,
    age_predictions
)

age_confounding_metrics = pd.DataFrame([{
    "n_controls": len(controls),
    "cv": "5-fold",
    "MAE_years": age_mae,
    "R2": age_r2
}])

display(age_confounding_metrics)

age_confounding_metrics.to_csv(
    OUTPUT_DIR / "age_prediction_confounding.csv",
    index=False,
    encoding="utf-8-sig"
)

,n_controls,cv,MAE_years,R2
0,128,5-fold,8.980406,0.498875


In [ ]:
#baseline для предсказания возраста
#создаем массив для baseline-прогнозов
baseline_predictions = np.full(
    len(controls),
    np.nan
)
#используем те же самые fold, что и для модели возраста
for train_idx, test_idx in cv.split(X_age):
    #наивный прогноз:медианный возраст обучающей выборки
    median_train_age = np.median(
        y_age[train_idx]
    )
    baseline_predictions[test_idx] = (
        median_train_age
    )
#считаем ошибку baseline
baseline_mae = mean_absolute_error(
    y_age,
    baseline_predictions
)
print(
    "MAE модели:",
    round(age_mae, 2)
)
print(
    "MAE baseline:",
    round(baseline_mae, 2)
)
print(
    "R2 модели:",
    round(age_r2, 3)
)

MAE модели: 8.98
MAE baseline: 14.55
R2 модели: 0.499


ЭЭГ-признаки продемонстрировали способность предсказывать возраст участников контрольной группы (5-fold CV, MAE = 8.98 года, R² = 0.499), что свидетельствует о наличии выраженной возраст-связанной информации в признаковом пространстве. Для оценки качества предсказания возраста результат модели был дополнительно сопоставлен с baseline-моделью, которая предсказывала медианный возраст обучающей выборки. MAE основной модели составила 8.98 года, тогда как MAE baseline — 14.55 года. Таким образом, использование EEG-признаков позволило заметно снизить ошибку предсказания возраста. Это показывает, что рассматриваемые EEG-признаки содержат информацию, связанную с возрастом, поэтому возраст может выступать потенциальным конфаундером при классификации ПТСР.Для оценки качества предсказания возраста результат модели был дополнительно сопоставлен с baseline-моделью, которая предсказывала медианный возраст обучающей выборки. MAE основной модели составила 8.98 года, тогда как MAE baseline — 14.55 года. Таким образом, использование EEG-признаков позволило заметно снизить ошибку предсказания возраста. Это показывает, что рассматриваемые EEG-признаки содержат информацию, связанную с возрастом, поэтому возраст может выступать потенциальным конфаундером при классификации ПТСР.

Одновременно в контрольной подгруппе 60+ была выявлена высокая доля ложноположительных классификаций PTSD (20/21; FPR = 95.2%, specificity = 4.8%).

Совокупность этих результатов указывает на потенциальное возрастное смещение классификатора и необходимость дополнительного контроля возраста при интерпретации его диагностических характеристик. При этом данные результаты не доказывают, что возраст является единственным или непосредственным источником ошибочной классификации.

In [ ]:
from scipy.stats import spearmanr
THRESHOLD = 0.5

In [ ]:
def age_group(age):
    if pd.isna(age):
        return np.nan
    elif age < 40:
        return "<40"
    elif 40 <= age < 50:
        return "40–49"
    elif 50 <= age < 60:
        return "50–59"
    elif 60 <= age < 65:
        return "60–64"
    else:
        return "65+"

control_age = df[
    (df["group"] == "Норма") &
    df["age"].notna() &
    df["ptsd_probability"].notna()
].copy()

control_age["age_group"] = control_age["age"].apply(age_group)

control_age["ptsd_prediction"] = (
    control_age["ptsd_probability"] >= THRESHOLD
).astype(int)

# 1. Возраст → вероятность ПТСР

rho, p_value = spearmanr(
    control_age["age"],
    control_age["ptsd_probability"]
)

age_probability_result = pd.DataFrame([{
    "n_controls": len(control_age),
    "spearman_rho": rho,
    "p_value": p_value,
    "threshold": THRESHOLD
}])

display(age_probability_result)

age_probability_result.to_csv(
    OUTPUT_DIR / "age_vs_ptsd_probability_spearman.csv",
    index=False,
    encoding="utf-8-sig"
)


# 2. FPR / specificity по возрастным группам

age_fpr_results = []

for age_group_name, subset in control_age.groupby(
    "age_group",
    dropna=True
):

    n = len(subset)

    fp = int(subset["ptsd_prediction"].sum())
    tn = n - fp

    fpr = fp / n if n > 0 else np.nan
    specificity = tn / n if n > 0 else np.nan

    age_fpr_results.append({
        "age_group": age_group_name,
        "n_controls": n,
        "false_positive_ptsd_n": fp,
        "true_negative_n": tn,
        "false_positive_rate": fpr,
        "specificity": specificity,
        "mean_ptsd_probability":
            subset["ptsd_probability"].mean(),
        "median_ptsd_probability":
            subset["ptsd_probability"].median()
    })

age_fpr = pd.DataFrame(age_fpr_results)

age_order = [
    "<40",
    "40–49",
    "50–59",
    "60–64",
    "65+"
]

age_fpr["age_group"] = pd.Categorical(
    age_fpr["age_group"],
    categories=age_order,
    ordered=True
)

age_fpr = age_fpr.sort_values("age_group")

display(age_fpr)

age_fpr.to_csv(
    OUTPUT_DIR / "age_fpr_specificity.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Распределение вероятности ПТСР по возрастным группам
age_probability_distribution = (
    control_age
    .groupby("age_group", dropna=True)["ptsd_probability"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

age_probability_distribution["age_group"] = pd.Categorical(
    age_probability_distribution["age_group"],
    categories=age_order,
    ordered=True
)

age_probability_distribution = (
    age_probability_distribution
    .sort_values("age_group")
)

display(age_probability_distribution)

age_probability_distribution.to_csv(
    OUTPUT_DIR / "ptsd_probability_by_age_group.csv",
    index=False,
    encoding="utf-8-sig"
)



# 4. ОТДЕЛЬНО: НОРМАЛЬНОЕ СТАРЕНИЕ 65+

normal_65 = control_age[
    control_age["age"] >= 65
].copy()

normal_65_summary = pd.DataFrame([{
    "age_group": "65+",
    "n": len(normal_65),
    "false_positive_ptsd_n":
        int(normal_65["ptsd_prediction"].sum()),
    "true_negative_n":
        int((normal_65["ptsd_prediction"] == 0).sum()),
    "false_positive_rate":
        (
            normal_65["ptsd_prediction"] == 1
        ).mean(),
    "specificity":
        (
            normal_65["ptsd_prediction"] == 0
        ).mean(),
    "mean_ptsd_probability":
        normal_65["ptsd_probability"].mean(),
    "median_ptsd_probability":
        normal_65["ptsd_probability"].median()
}])

display(normal_65_summary)

normal_65_summary.to_csv(
    OUTPUT_DIR / "specificity_normal_65plus.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. СПИСОК УЧАСТНИКОВ 65+

normal_65_detail = normal_65[
    [
        "subject_id",
        "age",
        "ptsd_probability",
        "ptsd_prediction"
    ]
].sort_values(
    "ptsd_probability",
    ascending=False
)

display(normal_65_detail)

normal_65_detail.to_csv(
    OUTPUT_DIR / "normal_65plus_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\nВсе результаты сохранены.")

,n_controls,spearman_rho,p_value,threshold
0,128,-0.058353,0.512941,0.5


,age_group,n_controls,false_positive_ptsd_n,true_negative_n,false_positive_rate,specificity,mean_ptsd_probability,median_ptsd_probability
3,<40,85,26,59,0.305882,0.694118,0.320330,0.039393
0,40–49,12,2,10,0.166667,0.833333,0.135047,0.028143
1,50–59,10,0,10,0.000000,1.000000,0.026941,0.015382
2,65+,21,20,1,0.952381,0.047619,0.908456,0.999623


,age_group,n,mean,median,std,min,max
3,<40,85,0.320330,0.039393,0.416756,0.000464,0.999920
0,40–49,12,0.135047,0.028143,0.258425,0.007570,0.815846
1,50–59,10,0.026941,0.015382,0.026825,0.003685,0.080941
2,65+,21,0.908456,0.999623,0.232153,0.000037,1.000000


,age_group,n,false_positive_ptsd_n,true_negative_n,false_positive_rate,specificity,mean_ptsd_probability,median_ptsd_probability
0,65+,21,20,1,0.952381,0.047619,0.908456,0.999623


,subject_id,age,ptsd_probability,ptsd_prediction
0,A003_68,68.0,1.000000,1
14,А016_65,65.0,0.999999,1
18,А022_68,68.0,0.999994,1
17,А020_67,67.0,0.999976,1
10,А010_67,67.0,0.999975,1
11,А011_69,69.0,0.999953,1
2,A037_69,69.0,0.999922,1
15,А017_67,67.0,0.999860,1
7,А007_68,68.0,0.999796,1
3,A038_67,67.0,0.999677,1



Все результаты сохранены.


**Специфичность на соматоформной группе**

Соматоформная группа используется для проверки того, является ли высокая вероятность ПТСР специфичной для ПТСР, а не возникает при любом отклонении от контрольной группы.

При пороге 0.5 рассчитываем долю ложноположительных ПТСР, специфичность и распределение предсказанных вероятностей.

In [12]:
#оставляем только соматоформную группу
somato = df[
    df["group"] == "Соматоформные"
].copy()

#при вероятности >= 0.5 модель относит участника к ПТСР
somato["ptsd_prediction"] = (
    somato["ptsd_probability"] >= THRESHOLD
).astype(int)

n_somato = len(somato)

#для соматоформной группы любой прогноз ПТСР считается ложноположительным
false_positive_somato = int(
    somato["ptsd_prediction"].sum()
)

true_negative_somato = (
    n_somato -
    false_positive_somato
)

somato_fpr = (
    false_positive_somato / n_somato
    if n_somato > 0
    else np.nan
)

somato_specificity = (
    true_negative_somato / n_somato
    if n_somato > 0
    else np.nan
)

somato_metrics = pd.DataFrame([{
    "group": "Соматоформные",
    "n": n_somato,
    "threshold": THRESHOLD,
    "false_positive_ptsd_n":
        false_positive_somato,
    "true_negative_n":
        true_negative_somato,
    "false_positive_rate":
        somato_fpr,
    "specificity":
        somato_specificity,
    "mean_ptsd_probability":
        somato["ptsd_probability"].mean(),
    "median_ptsd_probability":
        somato["ptsd_probability"].median(),
    "min_ptsd_probability":
        somato["ptsd_probability"].min(),
    "max_ptsd_probability":
        somato["ptsd_probability"].max()
}])

display(somato_metrics)

somato_metrics.to_csv(
    OUTPUT_DIR /
    "specificity_somatoform.csv",
    index=False,
    encoding="utf-8-sig"
)

,group,n,threshold,false_positive_ptsd_n,true_negative_n,false_positive_rate,specificity,mean_ptsd_probability,median_ptsd_probability,min_ptsd_probability,max_ptsd_probability
0,Соматоформные,63,0.5,43,20,0.68254,0.31746,0.710313,0.989245,0.004471,1.0


In [13]:
somato_high_probability = (
    somato[
        [
            "subject_id",
            "ptsd_probability",
            "ptsd_prediction"
        ]
    ]
    .sort_values(
        "ptsd_probability",
        ascending=False
    )
)

display(
    somato_high_probability.head(20)
)

,subject_id,ptsd_probability,ptsd_prediction
211,ZAQH1,1.000000,1
173,KREH6,1.000000,1
156,BLDY8,1.000000,1
153,ASDF9,1.000000,1
166,FWTU4,1.000000,1
157,CVUK7,1.000000,1
155,BLDY6,1.000000,1
181,MVCP4,0.999998,1
212,ZILO6,0.999998,1
209,XYZT9,0.999997,1


**Проверка влияния поведенческих показателей**

Проверяем, не сводится ли вероятность ПТСР к скорости выполнения таблиц Шульте.

Для этого анализируем связь поведенческих показателей с вероятностью ПТСР и отдельно оцениваем способность модели, использующей только поведенческие показатели, разделять ПТСР и контроль.

In [14]:
BEHAVIOR_FEATURES = [
    "behavior_S1_time",
    "behavior_S2_time",
    "behavior_S3_time",
    "behavior_S4_time",
    "behavior_S5_time",
    "behavior_mean_time",
    "behavior_median_time",
    "behavior_std_time",
    "behavior_delta_5_1",
    "behavior_time_slope"
]

available_behavior_features = [
    feature
    for feature in BEHAVIOR_FEATURES
    if feature in df.columns
]

print(
    "Доступные поведенческие признаки:",
    len(available_behavior_features)
)

print(
    available_behavior_features
)

Доступные поведенческие признаки: 10
['behavior_S1_time', 'behavior_S2_time', 'behavior_S3_time', 'behavior_S4_time', 'behavior_S5_time', 'behavior_mean_time', 'behavior_median_time', 'behavior_std_time', 'behavior_delta_5_1', 'behavior_time_slope']


In [15]:
behavior_correlation_results = []

for feature in available_behavior_features:

    subset = df[
        df[feature].notna()
        &
        df["ptsd_probability"].notna()
    ].copy()

    if len(subset) < 3:
        continue

    rho, p_value = spearmanr(
        subset[feature],
        subset["ptsd_probability"]
    )

    behavior_correlation_results.append({
        "feature": feature,
        "n": len(subset),
        "spearman_rho": rho,
        "abs_rho": abs(rho),
        "p_value": p_value
    })

behavior_correlations = pd.DataFrame(
    behavior_correlation_results
)

if len(behavior_correlations) > 0:

    behavior_correlations = (
        behavior_correlations
        .sort_values(
            "abs_rho",
            ascending=False
        )
    )

display(
    behavior_correlations
)

behavior_correlations.to_csv(
    OUTPUT_DIR /
    "behavior_vs_ptsd_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

,feature,n,spearman_rho,abs_rho,p_value
7,behavior_std_time,137,0.714112,0.714112,1.175062e-22
0,behavior_S1_time,165,0.375559,0.375559,6.673040e-07
9,behavior_time_slope,137,-0.305190,0.305190,2.874697e-04
5,behavior_mean_time,137,0.264963,0.264963,1.753894e-03
8,behavior_delta_5_1,144,-0.221432,0.221432,7.648202e-03
6,behavior_median_time,137,0.215764,0.215764,1.133431e-02
1,behavior_S2_time,146,0.152503,0.152503,6.611795e-02
4,behavior_S5_time,152,0.120560,0.120560,1.390079e-01
3,behavior_S4_time,148,0.084945,0.084945,3.046581e-01
2,behavior_S3_time,149,0.011323,0.011323,8.909863e-01


In [16]:
from sklearn.model_selection import (
    LeaveOneOut,
    cross_val_predict
)

from sklearn.linear_model import (
    LogisticRegression
)

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score
)

In [17]:
#используем только основную диагностическую пару ПТСР / Норма
behavior_binary = df[
    df["group"].isin(
        ["Норма", "ПТСР"]
    )
].copy()

behavior_binary["target"] = (
    behavior_binary["group"] == "ПТСР"
).astype(int)

X_behavior = behavior_binary[
    available_behavior_features
].copy()

y_behavior = behavior_binary[
    "target"
].to_numpy()


behavior_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE
        )
    )
])


#одна строка в таблице соответствует одному испытуемому,
#поэтому LeaveOneOut здесь эквивалентен LOSO
loo = LeaveOneOut()


behavior_probability = (
    cross_val_predict(
        behavior_pipeline,
        X_behavior,
        y_behavior,
        cv=loo,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]
)


behavior_prediction = (
    behavior_probability >= THRESHOLD
).astype(int)


behavior_auc = roc_auc_score(
    y_behavior,
    behavior_probability
)

behavior_balanced_accuracy = (
    balanced_accuracy_score(
        y_behavior,
        behavior_prediction
    )
)


behavior_only_metrics = pd.DataFrame([{
    "model": "behavior_only",
    "n": len(behavior_binary),
    "n_features":
        len(available_behavior_features),
    "roc_auc":
        behavior_auc,
    "balanced_accuracy":
        behavior_balanced_accuracy
}])


display(
    behavior_only_metrics
)

behavior_only_metrics.to_csv(
    OUTPUT_DIR /
    "behavior_only_model_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

,model,n,n_features,roc_auc,balanced_accuracy
0,behavior_only,153,10,0.942813,0.865312


In [18]:
if "behavior_mean_time" in df.columns:

    X_single_behavior = behavior_binary[
        ["behavior_mean_time"]
    ].copy()

    single_behavior_probability = (
        cross_val_predict(
            behavior_pipeline,
            X_single_behavior,
            y_behavior,
            cv=loo,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
    )

    single_behavior_prediction = (
        single_behavior_probability
        >= THRESHOLD
    ).astype(int)

    single_behavior_auc = (
        roc_auc_score(
            y_behavior,
            single_behavior_probability
        )
    )

    single_behavior_ba = (
        balanced_accuracy_score(
            y_behavior,
            single_behavior_prediction
        )
    )

    single_behavior_metrics = pd.DataFrame([{
        "model":
            "behavior_mean_time_only",
        "ROC_AUC":
            single_behavior_auc,
        "balanced_accuracy":
            single_behavior_ba
    }])

    display(
        single_behavior_metrics
    )

,model,ROC_AUC,balanced_accuracy
0,behavior_mean_time_only,0.79125,0.737969


Выводы:

1) Проверка специфичности для нормального старения не пройдена в текущем анализе. При пороге 0,5 модель демонстрирует крайне высокий уровень ложноположительных классификаций среди участников 65+.
2) Отсутствие общей корреляции с возрастом не снимает проблему. Результаты по возрастным группам показывают, что требуется отдельная проверка влияния возраста.
3) Причина пока не установлена. Эти результаты сами по себе не доказывают, что модель непосредственно использует возраст как признак. Ошибки могут быть связаны с возрастными особенностями ЭЭГ, составом выборки, различиями между обучающей и проверочной выборками или другими факторами.



Проверим, какие именно 20 участников 65+ получили высокую вероятность ПТСР.

Сравним ЭЭГ-признаки 20 участников 65+, которые получили высокую вероятность ПТСР, с участниками 50–59 лет и с обучающей выборкой.

Проверим, сохраняется ли эффект на независимой выборке.
Отдельно исследуем вклад каждого ЭЭГ-признака и устойчивость результатов при исключении возрастно-чувствительных признаков.
Проведем анализ чувствительности к порогу классификации, не подбирая его специально под эти же 21 наблюдение.

In [19]:

def extract_age_from_id(subject_id):
    if pd.isna(subject_id):
        return np.nan

    subject_id = str(subject_id)

    match = re.search(r"_(\d{2})$", subject_id)

    if match:
        return int(match.group(1))

    return np.nan


# Если age уже существует — используем его.
# Если age отсутствует — восстанавливаем для ID вида A003_68.
if "age" not in df.columns:

    df["age"] = df["subject_id"].apply(
        extract_age_from_id
    )

    print("Столбец age отсутствовал.")
    print("Возраст извлечён из subject_id для ID вида *_XX.")

else:

    print("Столбец age уже существует.")


# Проверяем результат

print("\nAge availability by group:")

age_check = (
    df.groupby("group")
    .agg(
        n=("subject_id", "count"),
        age_known=("age", lambda x: x.notna().sum()),
        age_missing=("age", lambda x: x.isna().sum())
    )
)

display(age_check)


# ============================================================
# Группа Норма с известным возрастом
# ============================================================

control_age = df[
    (df["group"] == "Норма") &
    df["age"].notna() &
    df["ptsd_probability"].notna()
].copy()


# ============================================================
# Возрастные группы
# ============================================================

def age_group(age):

    if pd.isna(age):
        return np.nan

    elif age < 40:
        return "<40"

    elif age < 50:
        return "40–49"

    elif age < 60:
        return "50–59"

    elif age < 65:
        return "60–64"

    else:
        return "65+"


control_age["age_group"] = (
    control_age["age"]
    .apply(age_group)
)

control_age["ptsd_prediction"] = (
    control_age["ptsd_probability"] >= 0.5
).astype(int)


print("\nControl age dataset:")
print("N =", len(control_age))

display(
    control_age[
        [
            "subject_id",
            "age",
            "age_group",
            "ptsd_probability",
            "ptsd_prediction"
        ]
    ].head()
)

print("\nAge groups:")

display(
    control_age["age_group"]
    .value_counts()
    .sort_index()
)

Столбец age уже существует.

Age availability by group:


,n,age_known,age_missing
group,,,
Норма,128,128,0
ПТСР,25,0,25
Соматоформные,63,0,63



Control age dataset:
N = 128


,subject_id,age,age_group,ptsd_probability,ptsd_prediction
0,A003_68,68.0,65+,1.000000,1
1,A036_69,69.0,65+,0.000037,0
2,A037_69,69.0,65+,0.999922,1
3,A038_67,67.0,65+,0.999677,1
4,А004_68,68.0,65+,0.999285,1



Age groups:


age_group
40–49    12
50–59    10
65+      21
<40      85
Name: count, dtype: int64

In [20]:
# ============================================================
# DIAGNOSTICS: AGE / 65+ SPECIFICITY / FEATURE EFFECTS
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path("/home/jupyter/project")
MODEL_PATH = ROOT / "model" / "final_model.joblib"
FEATURES_PATH = ROOT / "model" / "feature_names.json"

SOURCE_CSV = ROOT / "eeg_subject_level_features.csv"
PREDICTIONS_CSV = ROOT / "output" / "predictions.csv"

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.5
RANDOM_STATE = 42


# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

df_source = pd.read_csv(SOURCE_CSV)
df_pred = pd.read_csv(PREDICTIONS_CSV)

df = df_source.merge(
    df_pred,
    on="subject_id",
    how="inner"
)

df["target"] = (
    df["group"] == "ПТСР"
).astype(int)

with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    FEATURE_NAMES = json.load(f)

print("Dataset:", df.shape)
print("Features:", len(FEATURE_NAMES))


# ------------------------------------------------------------
# Age groups
# ------------------------------------------------------------
# ------------------------------------------------------------
# Age
# ------------------------------------------------------------

import re

# Проверяем, есть ли age
print("Columns containing 'age':")
print([c for c in df.columns if "age" in c.lower()])


# Если age отсутствует, извлекаем возраст из subject_id
if "age" not in df.columns:

    def extract_age_from_id(subject_id):
        if pd.isna(subject_id):
            return np.nan

        subject_id = str(subject_id).strip()

        # ID вида A003_68 -> 68
        match = re.search(r"_(\d{2})$", subject_id)

        if match:
            return int(match.group(1))

        return np.nan

    df["age"] = df["subject_id"].apply(
        extract_age_from_id
    )

    print("\nСоздан столбец age из subject_id.")

else:
    print("\nСтолбец age уже существует.")


# ------------------------------------------------------------
# Проверка возраста
# ------------------------------------------------------------

print("\nAge availability:")

age_check = (
    df.groupby("group")
    .agg(
        n=("subject_id", "count"),
        age_known=("age", lambda x: x.notna().sum()),
        age_missing=("age", lambda x: x.isna().sum())
    )
)

display(age_check)


# ------------------------------------------------------------
# Age groups
# ------------------------------------------------------------

def age_group(age):

    if pd.isna(age):
        return np.nan

    if age < 40:
        return "<40"

    if age < 50:
        return "40–49"

    if age < 60:
        return "50–59"

    if age < 65:
        return "60–64"

    return "65+"


# Только Норма с известным возрастом
control_age = df[
    (df["group"] == "Норма") &
    (df["age"].notna()) &
    (df["ptsd_probability"].notna())
].copy()


control_age["age_group"] = (
    control_age["age"]
    .apply(age_group)
)


control_age["ptsd_prediction"] = (
    control_age["ptsd_probability"] >= THRESHOLD
).astype(int)


# ------------------------------------------------------------
# Проверяем результат
# ------------------------------------------------------------

print("\nControl subjects with known age:")
print(len(control_age))

print("\nAge groups:")

display(
    control_age[
        "age_group"
    ].value_counts()
    .reindex(
        ["<40", "40–49", "50–59", "60–64", "65+"],
        fill_value=0
    )
)


print("\nExample:")

display(
    control_age[
        [
            "subject_id",
            "age",
            "age_group",
            "ptsd_probability",
            "ptsd_prediction"
        ]
    ].head(10)
)
# ============================================================
# 1. КАКИЕ ИМЕННО 65+ ПОЛУЧИЛИ ВЫСОКУЮ ВЕРОЯТНОСТЬ?
# ============================================================

normal_65 = control_age[
    control_age["age"] >= 65
].copy()

normal_65["error_type"] = np.where(
    normal_65["ptsd_prediction"] == 1,
    "false_positive_ptsd",
    "true_negative"
)

normal_65_detail = normal_65[
    [
        "subject_id",
        "age",
        "ptsd_probability",
        "ptsd_prediction",
        "error_type"
    ]
].sort_values(
    "ptsd_probability",
    ascending=False
)

print("\n===================================")
print("65+ PARTICIPANTS")
print("===================================")

display(normal_65_detail)

normal_65_detail.to_csv(
    OUTPUT_DIR / "normal_65plus_detailed_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


# Только 65+ с вероятностью >= 0.5

normal_65_fp = normal_65[
    normal_65["ptsd_probability"] >= THRESHOLD
].copy()

print("\n65+ false positives:", len(normal_65_fp))

display(normal_65_fp[
    [
        "subject_id",
        "age",
        "ptsd_probability"
    ]
].sort_values(
    "ptsd_probability",
    ascending=False
))


# ============================================================
# 2. СРАВНЕНИЕ EEG-ПРИЗНАКОВ:
#    65+ false positives vs 50–59
# ============================================================

age_50_59 = control_age[
    control_age["age_group"] == "50–59"
].copy()

age_65_fp_ids = set(
    normal_65_fp["subject_id"]
)

age_65_fp = df[
    df["subject_id"].isin(age_65_fp_ids)
].copy()

print("\n===================================")
print("FEATURE COMPARISON")
print("===================================")

print("65+ false positives:", len(age_65_fp))
print("50–59 controls:", len(age_50_59))


feature_comparison = []

for feature in FEATURE_NAMES:

    if feature not in df.columns:
        continue

    x65 = pd.to_numeric(
        age_65_fp[feature],
        errors="coerce"
    ).dropna()

    x50 = pd.to_numeric(
        age_50_59[feature],
        errors="coerce"
    ).dropna()

    if len(x65) == 0 or len(x50) == 0:
        continue

    mean_65 = x65.mean()
    mean_50 = x50.mean()

    pooled_sd = np.sqrt(
        (
            (len(x65) - 1) * x65.var(ddof=1) +
            (len(x50) - 1) * x50.var(ddof=1)
        )
        /
        (
            len(x65) + len(x50) - 2
        )
    )

    if pooled_sd > 0:
        standardized_difference = (
            mean_65 - mean_50
        ) / pooled_sd
    else:
        standardized_difference = np.nan

    feature_comparison.append({
        "feature": feature,
        "n_65plus_fp": len(x65),
        "n_50_59": len(x50),
        "mean_65plus_fp": mean_65,
        "mean_50_59": mean_50,
        "median_65plus_fp": x65.median(),
        "median_50_59": x50.median(),
        "standardized_difference":
            standardized_difference
    })

feature_comparison = pd.DataFrame(
    feature_comparison
)

feature_comparison["abs_standardized_difference"] = (
    feature_comparison["standardized_difference"]
    .abs()
)

feature_comparison = (
    feature_comparison
    .sort_values(
        "abs_standardized_difference",
        ascending=False
    )
)

display(
    feature_comparison.head(30)
)

feature_comparison.to_csv(
    OUTPUT_DIR / "65plus_fp_vs_50_59_features.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 3. КАКИЕ ПРИЗНАКИ СИЛЬНЕЕ ВСЕГО СВЯЗАНЫ С ВОЗРАСТОМ?
# ============================================================

age_feature_results = []

for feature in FEATURE_NAMES:

    if feature not in control_age.columns:
        continue

    x = pd.to_numeric(
        control_age[feature],
        errors="coerce"
    )

    mask = (
        x.notna() &
        control_age["age"].notna()
    )

    if mask.sum() < 10:
        continue

    rho_feature, p_feature = spearmanr(
        control_age.loc[mask, "age"],
        x.loc[mask]
    )

    age_feature_results.append({
        "feature": feature,
        "n": int(mask.sum()),
        "spearman_rho_age": rho_feature,
        "p_value": p_feature,
        "abs_rho": abs(rho_feature)
    })

age_feature_results = pd.DataFrame(
    age_feature_results
).sort_values(
    "abs_rho",
    ascending=False
)

print("\n===================================")
print("MOST AGE-RELATED EEG FEATURES")
print("===================================")

display(
    age_feature_results.head(30)
)

age_feature_results.to_csv(
    OUTPUT_DIR / "feature_age_correlations.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 4. КАКИЕ ПРИЗНАКИ ИСПОЛЬЗУЕТ ФИНАЛЬНАЯ МОДЕЛЬ?
# ============================================================

model = joblib.load(MODEL_PATH)

print("Pipeline steps:", model.named_steps)

selector = model.named_steps["selector"]

# автоматически находим классификатор
classifier = next(
    step for step in model.named_steps.values()
    if hasattr(step, "coef_") and hasattr(step, "predict_proba")
)

print("Classifier:", type(classifier).__name__)

selected_mask = selector.get_support()

selected_features = np.array(
    FEATURE_NAMES
)[selected_mask]

coefficients = classifier.coef_[0]

model_features = pd.DataFrame({
    "feature": selected_features,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
})

# Добавляем возрастную корреляцию

model_features = model_features.merge(
    age_feature_results[
        [
            "feature",
            "spearman_rho_age",
            "p_value"
        ]
    ],
    on="feature",
    how="left"
)

model_features = model_features.sort_values(
    "abs_coefficient",
    ascending=False
)

print("\n===================================")
print("MODEL SELECTED FEATURES")
print("===================================")

print(
    "Number selected:",
    len(model_features)
)

display(
    model_features.head(30)
)

model_features.to_csv(
    OUTPUT_DIR / "final_model_selected_features.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 5. НЕЗАВИСИМАЯ ПРОВЕРКА
#
# Финальная модель обучалась на 90 субъектах:
# 66 Норма + 24 ПТСР.
#
# Выделяем этих 90 из полного dataset и исключаем их
# из независимой проверки.
# ============================================================

training_subjects = df[
    (df["group"].isin(["Норма", "ПТСР"])) &
    (df["n_schulte_trials"] == 5) &
    (df["has_rest"] == True)
].copy()

# Проверка количества

print("\n===================================")
print("TRAINING SAMPLE RECONSTRUCTION")
print("===================================")

print(
    training_subjects["group"].value_counts()
)

print(
    "Training subjects:",
    training_subjects["subject_id"].nunique()
)


training_ids = set(
    training_subjects["subject_id"]
)

independent_controls = control_age[
    ~control_age["subject_id"].isin(training_ids)
].copy()

print("\nIndependent controls:", len(independent_controls))

# Независимая 65+

independent_65 = independent_controls[
    independent_controls["age"] >= 65
].copy()

independent_65["ptsd_prediction"] = (
    independent_65["ptsd_probability"] >= THRESHOLD
).astype(int)

independent_65_summary = pd.DataFrame([{
    "group": "Норма 65+ independent",
    "n": len(independent_65),
    "false_positive_ptsd_n":
        int(independent_65["ptsd_prediction"].sum()),
    "true_negative_n":
        int(
            (independent_65["ptsd_prediction"] == 0)
            .sum()
        ),
    "false_positive_rate":
        (
            independent_65["ptsd_prediction"] == 1
        ).mean(),
    "specificity":
        (
            independent_65["ptsd_prediction"] == 0
        ).mean(),
    "mean_ptsd_probability":
        independent_65["ptsd_probability"].mean(),
    "median_ptsd_probability":
        independent_65["ptsd_probability"].median()
}])

print("\n===================================")
print("INDEPENDENT 65+")
print("===================================")

display(independent_65_summary)

independent_65_summary.to_csv(
    OUTPUT_DIR / "independent_normal_65plus_specificity.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 6. АНАЛИЗ ЧУВСТВИТЕЛЬНОСТИ К ПОРОГУ
#
# Пороги задаются заранее и не оптимизируются на 65+.
# ============================================================

thresholds = [
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80
]

threshold_results = []

for threshold in thresholds:

    predictions_threshold = (
        normal_65["ptsd_probability"] >= threshold
    ).astype(int)

    fp = int(predictions_threshold.sum())
    tn = len(normal_65) - fp

    threshold_results.append({
        "threshold": threshold,
        "n_65plus": len(normal_65),
        "false_positive_ptsd_n": fp,
        "true_negative_n": tn,
        "false_positive_rate":
            fp / len(normal_65)
            if len(normal_65) > 0 else np.nan,
        "specificity":
            tn / len(normal_65)
            if len(normal_65) > 0 else np.nan,
        "mean_probability":
            normal_65["ptsd_probability"].mean(),
        "median_probability":
            normal_65["ptsd_probability"].median()
    })

threshold_sensitivity = pd.DataFrame(
    threshold_results
)

display(threshold_sensitivity)

threshold_sensitivity.to_csv(
    OUTPUT_DIR / "threshold_sensitivity_normal_65plus.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("FILES SAVED")
print("===================================")

print(
    "normal_65plus_detailed_predictions.csv"
)

print(
    "65plus_fp_vs_50_59_features.csv"
)

print(
    "feature_age_correlations.csv"
)

print(
    "final_model_selected_features.csv"
)

print(
    "independent_normal_65plus_specificity.csv"
)

print(
    "threshold_sensitivity_normal_65plus.csv"
)

print("\nГотово.")

Dataset: (216, 812)
Features: 252
Columns containing 'age':
[]

Создан столбец age из subject_id.

Age availability:


,n,age_known,age_missing
group,,,
Норма,128,128,0
ПТСР,25,0,25
Соматоформные,63,0,63



Control subjects with known age:
128

Age groups:


age_group
<40      85
40–49    12
50–59    10
60–64     0
65+      21
Name: count, dtype: int64


Example:


...

,feature,coefficient,abs_coefficient,spearman_rho_age,p_value
220,S1_Rest_T3_T4_delta_coherence,-0.512930,0.512930,-0.143339,1.853416e-01
11,Rest_T3_theta_rel,0.381200,0.381200,-0.328531,1.530355e-04
6,Rest_O1_beta_log,0.333042,0.333042,-0.042495,6.338808e-01
174,S1_Rest_O1_beta_log,-0.320534,0.320534,-0.536840,8.314128e-08
146,S1_T3_aperiodic_exponent,-0.307606,0.307606,0.210813,4.999832e-02
72,Rest_Fp1_Fp2_delta_asym,0.300362,0.300362,0.120191,1.765762e-01
80,Rest_O1_O2_alpha_asym,0.284017,0.284017,-0.393515,4.322538e-06
227,S1_Rest_O1_O2_beta_coherence,0.277087,0.277087,-0.031225,7.740335e-01
121,S1_T4_alpha_rel,0.274728,0.274728,0.254294,1.746332e-02
52,Rest_T3_T4_delta_coherence,0.274614,0.274614,-0.069240,4.373922e-01



TRAINING SAMPLE RECONSTRUCTION
group
Норма    66
ПТСР     24
Name: count, dtype: int64
Training subjects: 90

Independent controls: 62

INDEPENDENT 65+


,group,n,false_positive_ptsd_n,true_negative_n,false_positive_rate,specificity,mean_ptsd_probability,median_ptsd_probability
0,Норма 65+ independent,21,20,1,0.952381,0.047619,0.908456,0.999623


,threshold,n_65plus,false_positive_ptsd_n,true_negative_n,false_positive_rate,specificity,mean_probability,median_probability
0,0.2,21,20,1,0.952381,0.047619,0.908456,0.999623
1,0.3,21,20,1,0.952381,0.047619,0.908456,0.999623
2,0.4,21,20,1,0.952381,0.047619,0.908456,0.999623
3,0.5,21,20,1,0.952381,0.047619,0.908456,0.999623
4,0.6,21,19,2,0.904762,0.095238,0.908456,0.999623
5,0.7,21,19,2,0.904762,0.095238,0.908456,0.999623
6,0.8,21,19,2,0.904762,0.095238,0.908456,0.999623



FILES SAVED
normal_65plus_detailed_predictions.csv
65plus_fp_vs_50_59_features.csv
feature_age_correlations.csv
final_model_selected_features.csv
independent_normal_65plus_specificity.csv
threshold_sensitivity_normal_65plus.csv

Готово.


**Основные результаты**


В рамках анализа возрастного фактора было установлено, что индивидуальный возраст известен для всех 128 участников группы «Норма», поэтому дальнейшая проверка проводилась внутри контрольной группы. EEG-признаки содержат выраженную возрастную информацию: модель предсказания возраста получила MAE = 8.98 года и R² = 0.50, тогда как MAE baseline-модели составила 14.55 года. Кроме того, ряд EEG-признаков показал заметную корреляцию с возрастом. 
Наиболее выраженная проблема обнаружена в группе здоровых участников 65+: 20 из 21 были ошибочно классифицированы как ПТСР. Специфичность составила 4.8%, а FPR — 95.2%.Повышение порога классификации с 0.5 до 0.8 существенно не исправило ситуацию: ложноположительными остались 19 из 21 участников. 
 Возраст был существенно связан с рядом ЭЭГ_признаков. Наиболее сильная связь наблюдалась для Rest_T3_alpha_rel (ρ = −0.554) и S1_Rest_O1_beta_log (ρ = −0.537). Несколько признаков с выраженной возрастной корреляцией также имели заметные коэффициенты в финальной Logistic Regression.
Таким образом, результаты указывают на возраст как на потенциальный конфаундер. EEG-признаки содержат возрастную информацию, а модель демонстрирует крайне низкую специфичность на группе 65+, что ограничивает её переносимость на старшие возрастные группы. В дальнейшем необходимы более сбалансированная по возрасту обучающая выборка и дополнительная независимая валидация.